### Relevant imports & variables

In [2]:
import os
import re
import shutil
import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
# Constants
M_NH3 = 17.031  # Molar mass in g/mol
M_NH4 = 18.04

M_NO = 30.01
M_NO2 = 46.01
M_NO3 = 62.00
M_HNO2 = 47.01
M_HNO3 = 63.01
M_N = 14.007

M_SO4 = 96.06
M_SO2 = 64.066
M_S = 32.065

M_dry = 28.96
rho_dry = 1.225  # Density of dry air in kg/m³
Na = 6.022e23  # Avogadro's constant (mol⁻¹)

# Dictionary to pair pollutant and chemical element
sub_dict = {"NH3":"N" , "NH4":"N" ,
            "SO2":"S" , "SO4":"S",
            "NO":"N", "NO2":"N", "NO3":"N", "HNO2":"N", "HNO3":"N",}

# Dictionary to associate pollutants to groups
sub_groups = {
    "nhx": {"NH3", "NH4"},
    "nox": {"NO", "HNO3", "NO2", "NO3", "HNO2"},
    "sox": {"SO2", "SO4"},
}

# Names of the diagnostics
diag_types = ["emis", "ddep", "revp", "wdep", "wash", "conc", "fleo", "flns", "flud", "fl_to_S", "fl_to_N", "fl_to_O", "fl_to_E", "fl_to_D", "fl_to_U"]

# Current working directory
current_dir = os.getcwd()

# Pairing current and following months
next_month = {"01":"02",
              "02":"03",
              "03":"04",
              "04":"05",
              "05":"06",
              "06":"07",
              "07":"08",
              "08":"09",
              "09":"10",
              "10":"11",
              "11":"12",
              "12":"01"}

# Pairing current and following months when a crash occurred (comment the months already computed)
next_month_cut = {#"01":"02",
                  #"02":"03",
                  #"03":"04",
                  #"04":"05",
                  #"05":"06",
                  #"06":"07",
                  #"07":"08",
                  #"08":"09",
                  #"09":"10",
                  #"10":"11",
                  #"11":"12",
                  #"12":"01"
}

### Global mass balance calculation & correction

In [23]:
# Create a mass balance dataset for each group and months (next month is needed to compute the accumulation term)

# Create folder to store the result files
dossier = "mass_balance"
os.makedirs(dossier, exist_ok=True)

for month_A, month_B in next_month.items():
    # Open the datasets containing the diagnostics, averaged monthly and converted to kg atom per month
    
    file_A = xr.open_dataset(f'geos_diags_converted/geos_diags_converted_2019{month_A}.nc')
    file_B = xr.open_dataset(f'geos_diags_converted/geos_diags_converted_2019{month_B}.nc')
    
    results = {}
    
    # Loop over all pollutant groups
    for group_name, chems in sub_groups.items():
        
        # Define the right chemical element
        if group_name in ["nhx", "nox"]:
            atom = "N"
        elif group_name == "sox":
            atom = "S"
        else:
            continue
            
        # Extract the variable
        flns = file_A[f'kg_{atom}_flns_{group_name}']
        fleo = file_A[f'kg_{atom}_fleo_{group_name}']
        flud = file_A[f'kg_{atom}_flud_{group_name}']

        fl_to_N = file_A[f'kg_{atom}_fl_to_N_{group_name}']
        fl_to_S = file_A[f'kg_{atom}_fl_to_S_{group_name}']
        fl_to_E = file_A[f'kg_{atom}_fl_to_E_{group_name}']
        fl_to_O = file_A[f'kg_{atom}_fl_to_O_{group_name}']
        fl_to_U = file_A[f'kg_{atom}_fl_to_U_{group_name}']
        fl_to_D = file_A[f'kg_{atom}_fl_to_D_{group_name}']
        
        emis = file_A[f'kg_{atom}_emis_{group_name}']
        revp = file_A[f'kg_{atom}_revp_{group_name}']
        wdep = file_A[f'kg_{atom}_wdep_{group_name}']
        wash = file_A[f'kg_{atom}_wash_{group_name}']
        ddep = file_A[f'kg_{atom}_ddep_{group_name}']
    
        # === Input flux ===
        # Use shift and roll functions to take flows from neighbouring cells

        # North -> South
        fl_in_N = fl_to_S.shift(lat=-1).fillna(0)
        fl_in_S = fl_to_N.shift(lat=1).fillna(0)
        
        # East -> West
        fl_in_E = fl_to_O.roll(lon=-1, roll_coords=False).fillna(0)
        fl_in_O = fl_to_E.roll(lon=1, roll_coords=False).fillna(0)

        # Up -> Down
        fl_in_U = fl_to_D.fillna(0)
        fl_in_D = fl_to_U.shift(lev=-1).fillna(0)
    
        # Sum of input flows
        sum_fl_in = fl_in_N + fl_in_S + fl_in_E + fl_in_O + fl_in_U + fl_in_D
    
        # === Output flux ===
        
        fl_out_N = fl_to_N.fillna(0)
        fl_out_S = fl_to_S.fillna(0)
        fl_out_E = fl_to_E.fillna(0)
        fl_out_O = fl_to_O.fillna(0)
        fl_out_U = fl_to_U.fillna(0)
        fl_out_D = fl_to_D.fillna(0)
            
        sum_fl_out = fl_out_N + fl_out_S + fl_out_E + fl_out_O + fl_out_U + fl_out_D        
       
        # === Deposition and Reevaporation ===

        # --- Substracting reevaporation from deposition diagnostics
        wdep = wdep - revp
        wash = wash - revp

        # --- Check for negative values (negative deposition = reevaporation)
        neg_wdep = wdep.where(wdep < 0, 0) 
        neg_wash = wash.where(wash < 0, 0) 
        
        # --- Correct reevaporation (add the negative deposition to reevaporation and make reevaporation a positive value)
        revp_corrected = revp.copy()        
        revp_corrected = - (revp_corrected + neg_wdep + neg_wash)
        
        # --- Correct wet deposition (wdep < 0) et washout (wash < 0)
        wdep = wdep.where(wdep >= 0, 0)  # remplacer les négatifs par 0
        wash = wash.where(wash >= 0, 0)  # remplacer les négatifs par 0

        # --- Total deposition (= wet + dry, dry deposition is a 2d diagnostic)
        total_dep = wdep + wash
        total_dep.loc[dict(lev=total_dep.lev[46])] += ddep
       
        # === Accumulation ===

        # --- Difference in concentration between start and end of month
        conc_a = file_A[f'kg_{atom}_conc_{group_name}'].sel(time=f"2019-{month_A}-01")
        conc_b = file_B[f'kg_{atom}_conc_{group_name}'].sel(time=f"2019-{month_B}-01")
        accu = (conc_a - conc_b) / (31536000 / 12)
        # --- Assign the right time dimension
        accu = accu.expand_dims(time=[conc_a.time.values])
        
        # === Mass Balance ===
        equ_in = sum_fl_in + emis + revp_corrected
        equ_out = sum_fl_out + total_dep + accu

        equ = equ_in - equ_out
    
        # Store results before export
        results[group_name] = xr.Dataset({
            "flux_in": sum_fl_in,
            "emission": emis,
            "flux_out": sum_fl_out,
            "deposition": total_dep,
            "accumulation": accu,
            "reevaporation": revp_corrected,
            "equilibrium": equ,
            "flux_in_N": fl_in_N,
            "flux_in_S": fl_in_S,
            "flux_in_E": fl_in_E,
            "flux_in_O": fl_in_O,
            "flux_in_U": fl_in_U,
            "flux_in_D": fl_in_D,
            "flux_out_N": fl_out_N,
            "flux_out_S": fl_out_S,
            "flux_out_E": fl_out_E,
            "flux_out_O": fl_out_O,
            "flux_out_U": fl_out_U,
            "flux_out_D": fl_out_D
        })

    # === Exporting results ===
    for group_name, ds in results.items():
        ds.to_netcdf(f"mass_balance/mass_balance_{month_A}_{group_name}.nc")
        print(f"Saved : mass_balance/mass_balance_{month_A}_{group_name}.nc")

Sauvegardé : monthly/201901/mass_balance_01_nhx.nc
Sauvegardé : monthly/201901/mass_balance_01_nox.nc
Sauvegardé : monthly/201901/mass_balance_01_sox.nc
Sauvegardé : monthly/201902/mass_balance_02_nhx.nc
Sauvegardé : monthly/201902/mass_balance_02_nox.nc
Sauvegardé : monthly/201902/mass_balance_02_sox.nc
Sauvegardé : monthly/201903/mass_balance_03_nhx.nc
Sauvegardé : monthly/201903/mass_balance_03_nox.nc
Sauvegardé : monthly/201903/mass_balance_03_sox.nc
Sauvegardé : monthly/201904/mass_balance_04_nhx.nc
Sauvegardé : monthly/201904/mass_balance_04_nox.nc
Sauvegardé : monthly/201904/mass_balance_04_sox.nc
Sauvegardé : monthly/201905/mass_balance_05_nhx.nc
Sauvegardé : monthly/201905/mass_balance_05_nox.nc
Sauvegardé : monthly/201905/mass_balance_05_sox.nc
Sauvegardé : monthly/201906/mass_balance_06_nhx.nc
Sauvegardé : monthly/201906/mass_balance_06_nox.nc
Sauvegardé : monthly/201906/mass_balance_06_sox.nc
Sauvegardé : monthly/201907/mass_balance_07_nhx.nc
Sauvegardé : monthly/201907/mas

In [24]:
# Correcting global mass balance without touching the flows between cells (will be done later in "local corrections")

for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        
        # Charger les données du fichier
        filename = f"mass_balance_{month_A}_{group_name}"
        ds_pollutant = xr.open_dataset(f"mass_balance/{filename}.nc")
        
        flux_in = ds_pollutant["flux_in"].values
        flux_out = ds_pollutant["flux_out"].values
        emission = ds_pollutant["emission"].values
        deposition = ds_pollutant["deposition"].values
        accumulation = ds_pollutant["accumulation"].values
        reevaporation = ds_pollutant["reevaporation"].values

        equilibrium = ds_pollutant["equilibrium"].values

        ntime, nlev, nlat, nlon = flux_in.shape

        # Calculate global mass balance (without the flows = equation 1)

        emis_in = np.array([emission]).sum()
        revap_in = np.array([reevaporation]).sum()
        dep_out = np.array([deposition]).sum()
        accu_out = np.array([accumulation]).sum()
                
        input_glo = emis_in + revap_in
        output_glo = dep_out + accu_out
        equ_glo = input_glo - output_glo

        # Correcting global mass balance

        # Iterate over the whole grid
        for lev_cc in range(nlev):
            for lat_cc in range(nlat):
                for lon_cc in range(nlon):
                        
                    emis = emission[0, lev_cc, lat_cc, lon_cc]
                    revp = reevaporation[0, lev_cc, lat_cc, lon_cc]
                    dep = deposition[0, lev_cc, lat_cc, lon_cc]
                    accu = accumulation[0, lev_cc, lat_cc, lon_cc]
        
                    inputs = np.array([
                        emis,
                        revp
                    ])
                    sum_inputs = inputs.sum()
        
                    outputs = np.array([
                        dep,
                        accu,
                    ])
                    sum_outputs = outputs.sum()
                    
                    if sum_outputs == 0 :
                        continue
                    else:
                        # Define how local outputs contribute to total desequilibrium (only outputs to keep the same emissions values)
                        correction = (sum_outputs / output_glo) * equ_glo
                        # Defin how each local output contribute to local outputs
                        ratios = outputs / sum_outputs
                        # Define the correction that must be introduced in each local output
                        deltas = correction * ratios
                        # Add the correction to local outputs        
                        deposition[0, lev_cc, lat_cc, lon_cc] += deltas[0]
                        accumulation[0, lev_cc, lat_cc, lon_cc] += deltas[1]
                        # Update the equilibrium with new local output values 
                        equilibrium[0, lev_cc, lat_cc, lon_cc] = (flux_in[0, lev_cc, lat_cc, lon_cc] + 
                                                                  emis + revp - 
                                                                  flux_out[0, lev_cc, lat_cc, lon_cc] -
                                                                  deposition[0, lev_cc, lat_cc, lon_cc] -
                                                                  accumulation[0, lev_cc, lat_cc, lon_cc])

        # Calculate new global mass balance (sauf flux = equation 1)

        emis_in = np.array([emission]).sum()
        revap_in = np.array([reevaporation]).sum()
        dep_out = np.array([deposition]).sum()
        accu_out = np.array([accumulation]).sum()
           
        input_glo = emis_in + revap_in
        output_glo = dep_out + accu_out
        equ_glo = input_glo - output_glo
        
        # Display global equilibrium value for the month/group pair
        print(month_A, group_name, equ_glo)

        # Export results
        corrected_filename = f"mass_balance/corrected_{month_A}_{group_name}.nc"
        ds_corr = ds_pollutant.copy()

        ds_corr["deposition"].values = deposition
        ds_corr["accumulation"].values = accumulation
        ds_corr["equilibrium"].values = equilibrium

        ds_corr.to_netcdf(corrected_filename)
        print(f"Saved : {corrected_filename}")
        

01 nhx 1643.4854 31.772821 1679.8468 -4.588334944970328 1675.2582 1675.2584668128422 -0.0002881019047435984
Fichier corrigé enregistré : monthly/201901/corrected_01_nhx.nc
01 nox 1661.2885 15.417347 1685.4836 -8.777735525470307 1676.7058 1676.7059070526548 -9.650577976572094e-05
Fichier corrigé enregistré : monthly/201901/corrected_01_nox.nc
01 sox 1682.0868 69.39368 1744.7719 6.708430429061425 1751.4805 1751.480281014999 0.00018773500096358475
Fichier corrigé enregistré : monthly/201901/corrected_01_sox.nc
02 nhx 1797.5154 30.062405 1862.6106 -35.03260371684306 1827.5778 1827.5779919862819 -0.0002331972193587717
Fichier corrigé enregistré : monthly/201902/corrected_02_nhx.nc
02 nox 1633.5475 15.347967 1650.3414 -1.4458141460827174 1648.8955 1648.8956165179798 -0.0001087054797608289
Fichier corrigé enregistré : monthly/201902/corrected_02_nox.nc
02 sox 1608.203 61.53522 1668.4999 1.238486244268931 1669.7383 1669.7383641739564 -8.292395636999572e-05
Fichier corrigé enregistré : monthly/

### Local mass balance calculation & correction

In [25]:
# Functions to retrieve data and conduct local mass balance correction to 

# === Function to check mass balance and flows to and from each neighbouring cell ===
def retrieve_values(lev_cc, lat_cc, lon_cc,
                    flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                    flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                    emission,deposition,accumulation,reevaporation,equilibrium):  

    emis = emission[0, lev_cc, lat_cc, lon_cc]
    dep = deposition[0, lev_cc, lat_cc, lon_cc]
    accu = accumulation[0, lev_cc, lat_cc, lon_cc]
    revp = reevaporation[0, lev_cc, lat_cc, lon_cc]

    in_dirs = np.array([
        flux_in_N[0, lev_cc, lat_cc, lon_cc],
        flux_in_S[0, lev_cc, lat_cc, lon_cc],
        flux_in_E[0, lev_cc, lat_cc, lon_cc],
        flux_in_O[0, lev_cc, lat_cc, lon_cc],
        flux_in_U[0, lev_cc, lat_cc, lon_cc],
        flux_in_D[0, lev_cc, lat_cc, lon_cc],
    ])
    fl_in = in_dirs.sum()
    
    non_nul_in_fluxes = np.count_nonzero(in_dirs)
    
    out_dirs = np.array([
        flux_out_N[0, lev_cc, lat_cc, lon_cc],
        flux_out_S[0, lev_cc, lat_cc, lon_cc],
        flux_out_E[0, lev_cc, lat_cc, lon_cc],
        flux_out_O[0, lev_cc, lat_cc, lon_cc],
        flux_out_U[0, lev_cc, lat_cc, lon_cc],
        flux_out_D[0, lev_cc, lat_cc, lon_cc],
    ])
    fl_out = out_dirs.sum()

    non_nul_out_fluxes = np.count_nonzero(out_dirs)

    equ = (fl_in + emis + revp) - (fl_out + dep + accu)

    return (in_dirs, fl_in, non_nul_in_fluxes, out_dirs, fl_out, non_nul_out_fluxes, equ)

# === Function to check if any neighbour is the edge of the grid ===
def define_valid_dirs(lev_cc, lat_cc, lon_cc, nlev, nlat, nlon, corrections_coords):
    
    lat_n = lat_cc + 1 if lat_cc + 1 < nlat else None
    lat_s = lat_cc - 1 if lat_cc - 1 >= 0 else None
    lon_e = (lon_cc + 1) % nlon
    lon_w = (lon_cc - 1) % nlon
    lev_u = lev_cc - 1 if lev_cc - 1 >= 0 else None 
    lev_d = lev_cc + 1 if lev_cc + 1 < nlev else None

    valid_dirs = np.array([
        lat_n is not None and (lev_cc, lat_n, lon_cc) not in corrections_coords,  # N
        lat_s is not None and (lev_cc, lat_s, lon_cc) not in corrections_coords,  # S
        (lev_cc, lat_cc, lon_e) not in corrections_coords,  # E
        (lev_cc, lat_cc, lon_w) not in corrections_coords,  # O
        lev_u is not None and (lev_u, lat_cc, lon_cc) not in corrections_coords,  # U
        lev_d is not None and (lev_d, lat_cc, lon_cc) not in corrections_coords,  # D
    ])

    return(lat_n, lat_s, lon_e, lon_w, lev_u, lev_d, valid_dirs)

# === Function to update the equilibrium variable ===
def update_equ(lev_cc, lat_cc, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium):
    emis = emission[0, lev_cc, lat_cc, lon_cc]
    dep = deposition[0, lev_cc, lat_cc, lon_cc]
    accu = accumulation[0, lev_cc, lat_cc, lon_cc]
    revp = reevaporation[0, lev_cc, lat_cc, lon_cc]

    in_dirs = np.array([
        flux_in_N[0, lev_cc, lat_cc, lon_cc],
        flux_in_S[0, lev_cc, lat_cc, lon_cc],
        flux_in_E[0, lev_cc, lat_cc, lon_cc],
        flux_in_O[0, lev_cc, lat_cc, lon_cc],
        flux_in_U[0, lev_cc, lat_cc, lon_cc],
        flux_in_D[0, lev_cc, lat_cc, lon_cc],
    ])
    fl_in = in_dirs.sum()
        
    out_dirs = np.array([
        flux_out_N[0, lev_cc, lat_cc, lon_cc],
        flux_out_S[0, lev_cc, lat_cc, lon_cc],
        flux_out_E[0, lev_cc, lat_cc, lon_cc],
        flux_out_O[0, lev_cc, lat_cc, lon_cc],
        flux_out_U[0, lev_cc, lat_cc, lon_cc],
        flux_out_D[0, lev_cc, lat_cc, lon_cc],
    ])
    fl_out = out_dirs.sum()

    equ = (fl_in + emis + revp) - (fl_out + dep + accu)

    equilibrium[0, lev_cc, lat_cc, lon_cc] = equ

# === Function to correct a positive desequilibrium ===
def correc_posit_deseq(lev_cc, lat_cc, lon_cc,
                       flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                       flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                       flux_in, flux_out, emission,deposition,accumulation,reevaporation,equilibrium,
                       nlev, nlat, nlon, corrections_coords, non_nul_fluxes):
    
    (in_dirs, fl_in, non_nul_in_fluxes, out_dirs, fl_out, non_nul_out_fluxes, equ) = retrieve_values(lev_cc, lat_cc, lon_cc,
                                                              flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                                                              flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                                                              emission,deposition,accumulation,reevaporation,equilibrium)

    if non_nul_fluxes != non_nul_out_fluxes:
        return False
    if abs(equ) < SEUIL:
        return True
        
    (lat_n, lat_s, lon_e, lon_w, lev_u, lev_d, valid_dirs) = define_valid_dirs(lev_cc, lat_cc, lon_cc, nlev, nlat, nlon, corrections_coords)
    
    correction = abs(equ)
    masked_out_dirs = out_dirs * valid_dirs
    total_valid_out = masked_out_dirs.sum()
    
    if total_valid_out > 0:
        ratios = masked_out_dirs / total_valid_out
        deltas = correction * ratios
        
        if (deltas < 0).any():
            print(f"⚠️ Negative deltas in {(lev_cc, lat_cc, lon_cc)}")
            return False
        if not np.isclose(ratios.sum(), 1.0, atol=1e-6):
            print(f"⚠️ : Sum of ratios = {ratios.sum()} (Should be 1.0) {lev_cc, lat_cc, lon_cc}")
            return False
        if np.isnan(ratios).any():
            print(f"Nan ratios in {lev_cc, lat_cc, lon_cc}")
            return False

        if valid_dirs[0] and lat_n is not None and (lev_cc, lat_n, lon_cc) not in corrections_coords:
            flux_in_S[0, lev_cc, lat_n, lon_cc] += deltas[0] # = flux_in_N
            flux_in[0, lev_cc, lat_n, lon_cc] += deltas[0]
            update_equ(lev_cc, lat_n, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_out_N[0, lev_cc, lat_cc, lon_cc] += deltas[0]
        if valid_dirs[1] and lat_s is not None and (lev_cc, lat_s, lon_cc) not in corrections_coords:
            flux_in_N[0, lev_cc, lat_s, lon_cc] += deltas[1] # = flux_in_S
            flux_in[0, lev_cc, lat_s, lon_cc] += deltas[1]
            update_equ(lev_cc, lat_s, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_out_S[0, lev_cc, lat_cc, lon_cc] += deltas[1]
        if valid_dirs[2] and lon_e is not None and (lev_cc, lat_cc, lon_e) not in corrections_coords:
            flux_in_O[0, lev_cc, lat_cc, lon_e] += deltas[2] # = flux_in_E
            flux_in[0, lev_cc, lat_cc, lon_e] += deltas[2]
            update_equ(lev_cc, lat_cc, lon_e,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_out_E[0, lev_cc, lat_cc, lon_cc] += deltas[2]
        if valid_dirs[3] and lon_w is not None and (lev_cc, lat_cc, lon_w) not in corrections_coords:
            flux_in_E[0, lev_cc, lat_cc, lon_w] += deltas[3] # = flux_in_O
            flux_in[0, lev_cc, lat_cc, lon_w] += deltas[3]
            update_equ(lev_cc, lat_cc, lon_w,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_out_O[0, lev_cc, lat_cc, lon_cc] += deltas[3]
        if valid_dirs[4] and lev_u is not None and (lev_u, lat_cc, lon_cc) not in corrections_coords:
            flux_in_D[0, lev_u, lat_cc, lon_cc] += deltas[4] # = flux_in_U
            flux_in[0, lev_u, lat_cc, lon_cc] += deltas[4]
            update_equ(lev_u, lat_cc, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_out_U[0, lev_cc, lat_cc, lon_cc] += deltas[4]
        if valid_dirs[5] and lev_d is not None and (lev_d, lat_cc, lon_cc) not in corrections_coords:
            flux_in_U[0, lev_d, lat_cc, lon_cc] += deltas[5] # = flux_in_D
            flux_in[0, lev_d, lat_cc, lon_cc] += deltas[5]
            update_equ(lev_d, lat_cc, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_out_D[0, lev_cc, lat_cc, lon_cc] += deltas[5]            
        
        flux_out[0, lev_cc, lat_cc, lon_cc] += deltas.sum()
        update_equ(lev_cc, lat_cc, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
        return True

# === Function to correct a negative desequilibrium ===
def correc_negat_deseq(lev_cc, lat_cc, lon_cc,
                       flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                       flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                       flux_in, flux_out, emission,deposition,accumulation,reevaporation,equilibrium, 
                       nlev, nlat, nlon, corrections_coords, non_nul_fluxes):
    
    (in_dirs, fl_in, non_nul_in_fluxes, out_dirs, fl_out, non_nul_out_fluxes, equ) = retrieve_values(lev_cc, lat_cc, lon_cc,
                                                              flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                                                              flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                                                              emission,deposition,accumulation,reevaporation,equilibrium)
    if non_nul_fluxes != non_nul_in_fluxes:
        return False
    if abs(equ) < SEUIL:
        return True
        
    (lat_n, lat_s, lon_e, lon_w, lev_u, lev_d, valid_dirs) = define_valid_dirs(lev_cc, lat_cc, lon_cc, nlev, nlat, nlon, corrections_coords)
    
    correction = abs(equ)
    masked_in_dirs = in_dirs * valid_dirs
    total_valid_in = masked_in_dirs.sum()
    
    if total_valid_in > 0:
        ratios = masked_in_dirs / total_valid_in
        deltas = correction * ratios
        
        if (deltas < 0).any():
            print(f"⚠️ Negative deltas in {(lev_cc, lat_cc, lon_cc)}")
            return False
        if not np.isclose(ratios.sum(), 1.0, atol=1e-6) :
            print(f"⚠️ : Sum of ratios = {ratios.sum()} (should be 1.0) {lev_cc, lat_cc, lon_cc}")
            return False
        if np.isnan(ratios).any():
            print(f"Nan ratios in {lev_cc, lat_cc, lon_cc}")
            return False

        if valid_dirs[0] and lat_n is not None and (lev_cc, lat_n, lon_cc) not in corrections_coords:
            flux_out_S[0, lev_cc, lat_n, lon_cc] += deltas[0] # = flux_in_N
            flux_out[0, lev_cc, lat_n, lon_cc] += deltas[0]
            update_equ(lev_cc, lat_n, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_in_N[0, lev_cc, lat_cc, lon_cc] += deltas[0]
        if valid_dirs[1] and lat_s is not None and (lev_cc, lat_s, lon_cc) not in corrections_coords:
            flux_out_N[0, lev_cc, lat_s, lon_cc] += deltas[1] # = flux_in_S
            flux_out[0, lev_cc, lat_s, lon_cc] += deltas[1]
            update_equ(lev_cc, lat_s, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_in_S[0, lev_cc, lat_cc, lon_cc] += deltas[1]
        if valid_dirs[2] and lon_e is not None and (lev_cc, lat_cc, lon_e) not in corrections_coords:
            flux_out_O[0, lev_cc, lat_cc, lon_e] += deltas[2] # = flux_in_E
            flux_out[0, lev_cc, lat_cc, lon_e] += deltas[2]
            update_equ(lev_cc, lat_cc, lon_e,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_in_E[0, lev_cc, lat_cc, lon_cc] += deltas[2]
        if valid_dirs[3] and lon_w is not None and (lev_cc, lat_cc, lon_w) not in corrections_coords:
            flux_out_E[0, lev_cc, lat_cc, lon_w] += deltas[3] # = flux_in_O
            flux_out[0, lev_cc, lat_cc, lon_w] += deltas[3]
            update_equ(lev_cc, lat_cc, lon_w,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_in_O[0, lev_cc, lat_cc, lon_cc] += deltas[3]
        if valid_dirs[4] and lev_u is not None and (lev_u, lat_cc, lon_cc) not in corrections_coords:
            flux_out_D[0, lev_u, lat_cc, lon_cc] += deltas[4] # = flux_in_U
            flux_out[0, lev_u, lat_cc, lon_cc] += deltas[4]
            update_equ(lev_u, lat_cc, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_in_U[0, lev_cc, lat_cc, lon_cc] += deltas[4]
        if valid_dirs[5] and lev_d is not None and (lev_d, lat_cc, lon_cc) not in corrections_coords:
            flux_out_U[0, lev_d, lat_cc, lon_cc] += deltas[5] # = flux_in_D
            flux_out[0, lev_d, lat_cc, lon_cc] += deltas[5]
            update_equ(lev_d, lat_cc, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
            flux_in_D[0, lev_cc, lat_cc, lon_cc] += deltas[5]

        flux_in[0, lev_cc, lat_cc, lon_cc] += deltas.sum()
        update_equ(lev_cc, lat_cc, lon_cc,
               flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
               flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
               emission,deposition,accumulation,reevaporation,equilibrium)
        return True

# === Non-recursive mass balance correction algorithm ===
def nr_algo_correc(SEUIL, corrections_coords,
                flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                flux_in,flux_out,emission,deposition,accumulation,reevaporation,equilibrium,
                ntime, nlev, nlat, nlon):

    nb_iteration = 0
    
    cells_total = nlev * nlon * nlat

    nb_iteration +=1

    # Check for already equilibrated cells and add them to the list
    for lev_cc in range(nlev):
        for lat_cc in reversed(range(nlat)):
            for lon_cc in reversed(range(nlon)):
                (in_dirs, fl_in, non_nul_in_fluxes, out_dirs, fl_out, non_nul_out_fluxes, equ) = retrieve_values(lev_cc, lat_cc, lon_cc,
                                                          flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                                                          flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                                                          emission,deposition,accumulation,reevaporation,equilibrium)
                if abs(equ) < SEUIL:
                    corrections_coords.add((lev_cc, lat_cc, lon_cc))

    # Correction of the elements with the fewest non-zero fluxes first, 
    # to avoid preventing subsequent corrections when an upstream neighboring grid cell has already corrected its non-zero fluxes. 
    for non_nul_fluxes in range(6):
        for lev_cc in range(nlev):
            for lat_cc in reversed(range(nlat)):
                for lon_cc in reversed(range(nlon)):
                    # Check if element was already corrected
                    if (lev_cc, lat_cc, lon_cc) in corrections_coords:
                        continue
                    (in_dirs, fl_in, non_nul_in_fluxes, out_dirs, fl_out, non_nul_out_fluxes, equ) = retrieve_values(lev_cc, lat_cc, lon_cc,
                                                          flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                                                          flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                                                          emission,deposition,accumulation,reevaporation,equilibrium)
                    corrected = False    

                    # Check the sign of the desequilibrium to act either on the input or output flows
                    
                    if equ < -SEUIL:
                        corrected = correc_negat_deseq(lev_cc, lat_cc, lon_cc,
                                           flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                                           flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                                           flux_in, flux_out, emission,deposition,accumulation,reevaporation,equilibrium,
                                           nlev, nlat, nlon, corrections_coords, non_nul_fluxes)
                    if equ > SEUIL:

                        corrected = correc_posit_deseq(lev_cc, lat_cc, lon_cc,
                                           flux_in_N, flux_in_S,flux_in_E,flux_in_O, flux_in_U,flux_in_D,
                                           flux_out_N,flux_out_S,flux_out_E,flux_out_O,flux_out_U,flux_out_D,
                                           flux_in, flux_out, emission,deposition,accumulation,reevaporation,equilibrium, 
                                           nlev, nlat, nlon, corrections_coords, non_nul_fluxes)

                    # Add to the set of corrected elements
                    if corrected :
                        corrections_coords.add((lev_cc, lat_cc, lon_cc))

    return (
        flux_in_N, flux_in_S, flux_in_E, flux_in_O, flux_in_U, flux_in_D,
        flux_out_N, flux_out_S, flux_out_E, flux_out_O, flux_out_U, flux_out_D,
        flux_in, flux_out, emission, deposition, accumulation, reevaporation, equilibrium, corrections_coords
    )


In [26]:
# Local mass balance correction
for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        # Load file
        filename = f"corrected_{month_A}_{group_name}"
        ds_pollutant = xr.open_dataset(f"mass_balance/{filename}.nc")
        
        # Storing in numpys
        
        i_flux_in = ds_pollutant["flux_in"].values
        i_flux_out = ds_pollutant["flux_out"].values
        i_emission = ds_pollutant["emission"].values
        i_deposition = ds_pollutant["deposition"].values
        i_accumulation = ds_pollutant["accumulation"].values
        i_reevaporation = ds_pollutant["reevaporation"].values
        i_equilibrium = ds_pollutant["equilibrium"].values
        
        i_flux_in_N = ds_pollutant["flux_in_N"].values
        i_flux_in_S = ds_pollutant["flux_in_S"].values
        i_flux_in_E = ds_pollutant["flux_in_E"].values
        i_flux_in_O = ds_pollutant["flux_in_O"].values
        i_flux_in_U = ds_pollutant["flux_in_U"].values
        i_flux_in_D = ds_pollutant["flux_in_D"].values
        
        i_flux_out_N = ds_pollutant["flux_out_N"].values
        i_flux_out_S = ds_pollutant["flux_out_S"].values
        i_flux_out_E = ds_pollutant["flux_out_E"].values
        i_flux_out_O = ds_pollutant["flux_out_O"].values
        i_flux_out_U = ds_pollutant["flux_out_U"].values
        i_flux_out_D = ds_pollutant["flux_out_D"].values
        
        ntime, nlev, nlat, nlon = i_flux_in_N.shape
        
        i_in_dirs = np.array([
            i_flux_in_N,
            i_flux_in_S,
            i_flux_in_E,
            i_flux_in_O,
            i_flux_in_U,
            i_flux_in_D,
        ])
        
        i_out_dirs = np.array([
            i_flux_out_N,
            i_flux_out_S,
            i_flux_out_E,
            i_flux_out_O,
            i_flux_out_U,
            i_flux_out_D,
        ])
        
        # Create input flows in grid elements which don't have any nonzero input flows to allow local correction   
        nb_corrections = 0
        for lev_cc in range(nlev):
            for lat_cc in reversed(range(nlat)):
                for lon_cc in reversed(range(nlon)):
                    
                    emis = i_emission[0, lev_cc, lat_cc, lon_cc]
                    dep = i_deposition[0, lev_cc, lat_cc, lon_cc]
                    accu = i_accumulation[0, lev_cc, lat_cc, lon_cc]
                    revp = i_reevaporation[0, lev_cc, lat_cc, lon_cc]
                    equi = i_equilibrium[0, lev_cc, lat_cc, lon_cc]
        
                    in_dirs = np.array([
                        i_flux_in_N[0, lev_cc, lat_cc, lon_cc],
                        i_flux_in_S[0, lev_cc, lat_cc, lon_cc],
                        i_flux_in_E[0, lev_cc, lat_cc, lon_cc],
                        i_flux_in_O[0, lev_cc, lat_cc, lon_cc],
                        i_flux_in_U[0, lev_cc, lat_cc, lon_cc],
                        i_flux_in_D[0, lev_cc, lat_cc, lon_cc],
                    ])
                    fl_in = in_dirs.sum()
                    
                    out_dirs = np.array([
                        i_flux_out_N[0, lev_cc, lat_cc, lon_cc],
                        i_flux_out_S[0, lev_cc, lat_cc, lon_cc],
                        i_flux_out_E[0, lev_cc, lat_cc, lon_cc],
                        i_flux_out_O[0, lev_cc, lat_cc, lon_cc],
                        i_flux_out_U[0, lev_cc, lat_cc, lon_cc],
                        i_flux_out_D[0, lev_cc, lat_cc, lon_cc],
                    ])
                    fl_out = out_dirs.sum()
        
                    equ_reel = (fl_in + emis + revp) - (fl_out + dep + accu)
                    
                    if fl_in == 0:
                        correction = abs(equ_reel)
                        
                        # Check valid directions = neighbour is not the edge of the grid
                        lat_n = lat_cc + 1 if lat_cc + 1 < nlat else None
                        lat_s = lat_cc - 1 if lat_cc - 1 >= 0 else None
                        lon_e = (lon_cc + 1) % nlon
                        lon_w = (lon_cc - 1) % nlon
                        lev_u = lev_cc - 1 if lev_cc - 1 >= 0 else None
                        lev_d = lev_cc + 1 if lev_cc + 1 < nlev else None
                                        
                        valid_dirs = np.array([
                            out_dirs[0] != 0,  # N
                            out_dirs[1] != 0,  # S
                            out_dirs[2] != 0,  # E
                            out_dirs[3] != 0,  # O
                            out_dirs[4] != 0,  # U
                            out_dirs[5] != 0,  # D
                        ])

                        # Introduce new input flows based on existing output flows from the grid element
                        # Output flows from neighbours are adjusted consequently
                        masked_out_dirs = out_dirs * valid_dirs
                        total_valid_out = masked_out_dirs.sum()

                        if total_valid_out > 0:
                            ratios = masked_out_dirs / total_valid_out
                            deltas = correction * ratios
        
                            if not np.isclose(ratios.sum(), 1.0, atol=1e-6):
                                print(f"Avertissement : somme des ratios = {ratios.sum()} (devrait être 1.0)")
        
                            if valid_dirs[0] and lat_n is not None:
                                i_flux_out_S[0, lev_cc, lat_n, lon_cc] += deltas[0] # = flux_in_N
                                i_flux_out[0, lev_cc, lat_n, lon_cc] += deltas[0]
                                i_equilibrium[0, lev_cc, lat_n, lon_cc] -= deltas[0]
                                i_flux_in_N[0, lev_cc, lat_cc, lon_cc] += deltas[0]
                            if valid_dirs[1] and lat_s is not None:
                                i_flux_out_N[0, lev_cc, lat_s, lon_cc] += deltas[1] # = flux_in_S
                                i_flux_out[0, lev_cc, lat_s, lon_cc] += deltas[1]
                                i_equilibrium[0, lev_cc, lat_s, lon_cc] -= deltas[1]
                                i_flux_in_S[0, lev_cc, lat_cc, lon_cc] += deltas[1]
                            if valid_dirs[2] and lon_e is not None:
                                i_flux_out_O[0, lev_cc, lat_cc, lon_e] += deltas[2] # = flux_in_E
                                i_flux_out[0, lev_cc, lat_cc, lon_e] += deltas[2]
                                i_equilibrium[0, lev_cc, lat_cc, lon_e] -= deltas[2]
                                i_flux_in_E[0, lev_cc, lat_cc, lon_cc] += deltas[2]
                            if valid_dirs[3] and lon_w is not None:
                                i_flux_out_E[0, lev_cc, lat_cc, lon_w] += deltas[3] # = flux_in_O
                                i_flux_out[0, lev_cc, lat_cc, lon_w] += deltas[3]
                                i_equilibrium[0, lev_cc, lat_cc, lon_w] -= deltas[3]
                                i_flux_in_O[0, lev_cc, lat_cc, lon_cc] += deltas[3]
                            if valid_dirs[4] and lev_u is not None:
                                i_flux_out_D[0, lev_u, lat_cc, lon_cc] += deltas[4] # = flux_in_U
                                i_flux_out[0, lev_u, lat_cc, lon_cc] += deltas[4]
                                i_equilibrium[0, lev_u, lat_cc, lon_cc] -= deltas[4]
                                i_flux_in_U[0, lev_cc, lat_cc, lon_cc] += deltas[4]
                            if valid_dirs[5] and lev_d is not None:
                                i_flux_out_U[0, lev_d, lat_cc, lon_cc] += deltas[5] # = flux_in_D
                                i_flux_out[0, lev_d, lat_cc, lon_cc] += deltas[5]
                                i_equilibrium[0, lev_d, lat_cc, lon_cc] -= deltas[5]
                                i_flux_in_D[0, lev_cc, lat_cc, lon_cc] += deltas[5]

                            # Update sum variables
                            i_flux_in[0, lev_cc, lat_cc, lon_cc] += deltas.sum()
                            i_equilibrium[0, lev_cc, lat_cc, lon_cc] += deltas.sum()
        
        
        # Correction thresholds - conversion from /year to /second
        if group_name == "nhx" :
            SEUIL = 15 / 3.156e7
        if group_name == "nox" :
            SEUIL = 20 / 3.156e7
        if group_name == "sox" :
            SEUIL = 40 / 3.156e7

        # Set to store corrected grid elements
        i_corrections_coords = set()

        # Call to the mass balance correction function
        (
            c_flux_in_N, c_flux_in_S, c_flux_in_E, c_flux_in_O, c_flux_in_U, c_flux_in_D,
            c_flux_out_N, c_flux_out_S, c_flux_out_E, c_flux_out_O, c_flux_out_U, c_flux_out_D,
            c_flux_in, c_flux_out, c_emission, c_deposition, c_accumulation, c_reevaporation, c_equilibrium, c_corrections_coords
        ) = nr_algo_correc(
            SEUIL, i_corrections_coords,
            i_flux_in_N, i_flux_in_S, i_flux_in_E, i_flux_in_O, i_flux_in_U, i_flux_in_D,
            i_flux_out_N, i_flux_out_S, i_flux_out_E, i_flux_out_O, i_flux_out_U, i_flux_out_D,
            i_flux_in, i_flux_out, i_emission, i_deposition, i_accumulation, i_reevaporation, i_equilibrium,
            ntime, nlev, nlat, nlon
        )
        points = []

        # === Check the number of corrected elements ===
        
        # print(len(c_corrections_coords))
        # # Check in the files if the number is correct        
        # for lev_cc in range(nlev):
        #     for lat_cc in reversed(range(nlat)):
        #         for lon_cc in reversed(range(nlon)):
            
        #             eq_val = c_equilibrium[0, lev_cc, lat_cc, lon_cc]
        
        #             if eq_val > SEUIL:
        #                 points.append((lev_cc, lat_cc, lon_cc, eq_val))
        #             elif eq_val < -SEUIL:
        #                 points.append((lev_cc, lat_cc, lon_cc, eq_val))
        #             else:
        #                 c_corrections_coords.discard((lev_cc, lat_cc, lon_cc))
        
        # # Check the number of balanced elements
        # print(len(c_corrections_coords))
        
        # # Export to check the unbalanced elements
        # df = pd.DataFrame(points, columns=["lev", "lat", "lon", "equilibrium"])
        # print(f"{filename} : {len(points)} cells modified, {len(c_corrections_coords)} cells unbalanced")
        # df.to_csv(f"mass_balance/equilibrium_nr_{testfilename}{month_A}.csv", index=False)
        # print(f"✅ Saved in equilibrium_nr_{testfilename}.csv")

        # Export the diagnostics after mass balance correction
        corrected_filename = f"equilibrated_{month_A}_{group_name}.nc"

        ds_corr = ds_pollutant.copy()
                
        ds_corr["flux_in"].values = c_flux_in        
        ds_corr["flux_out"].values = c_flux_out                
        ds_corr["equilibrium"].values = c_equilibrium
        
        ds_corr["flux_in_N"].values = c_flux_in_N        
        ds_corr["flux_in_S"].values = c_flux_in_S        
        ds_corr["flux_in_E"].values = c_flux_in_E        
        ds_corr["flux_in_O"].values = c_flux_in_O        
        ds_corr["flux_in_U"].values = c_flux_in_U        
        ds_corr["flux_in_D"].values = c_flux_in_D                
        
        ds_corr["flux_out_N"].values = c_flux_out_N        
        ds_corr["flux_out_S"].values = c_flux_out_S        
        ds_corr["flux_out_E"].values = c_flux_out_E        
        ds_corr["flux_out_O"].values = c_flux_out_O        
        ds_corr["flux_out_U"].values = c_flux_out_U        
        ds_corr["flux_out_D"].values = c_flux_out_D
                
        ds_corr.to_netcdf(f"mass_balance/{corrected_filename}")
        print(f"Saved : {corrected_filename}")

604373
462348
106
corrected_01_nhx : 153612 cells not equilibrated, 106 cells corrected
✅ Données enregistrées dans equilibrium_nr_corrected_01_nhx.csv
Fichier corrigé enregistré : monthly/201901/NE_equilibrated_01_nhx.nc
610465
443716
1590
corrected_01_nox : 173752 cells not equilibrated, 1590 cells corrected
✅ Données enregistrées dans equilibrium_nr_corrected_01_nox.csv
Fichier corrigé enregistré : monthly/201901/NE_equilibrated_01_nox.nc
606922
482896
10
corrected_01_sox : 132992 cells not equilibrated, 10 cells corrected
✅ Données enregistrées dans equilibrium_nr_corrected_01_sox.csv
Fichier corrigé enregistré : monthly/201901/NE_equilibrated_01_sox.nc
604011
491630
147
corrected_02_nhx : 124360 cells not equilibrated, 147 cells corrected
✅ Données enregistrées dans equilibrium_nr_corrected_02_nhx.csv
Fichier corrigé enregistré : monthly/201902/NE_equilibrated_02_nhx.nc
609419
469714
2503
corrected_02_nox : 148661 cells not equilibrated, 2503 cells corrected
✅ Données enregistrées

### Transport & deposition

In [27]:
# Preparing files

# Create folder to store the prepared files
dossier = "srm_computation_files"
os.makedirs(dossier, exist_ok=True)

for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        # load file
        filename = f"equilibrated_{month_A}_{group_name}"
        ds_polluant = xr.open_dataset(f"mass_balance/{filename}.nc")
        
        # === Agregate altitude levels above 31 to improve calculation time (0 to 15 in the .nc files) ===

        # Sum all flows in a new table and reset the lev dimension
        first_level = ds_polluant.sel(lev=slice(ds_polluant.lev[0], ds_polluant.lev[16])).sum(dim="lev")
        first_level = first_level.expand_dims(dim={"lev": [ds_polluant.lev[0].item()]}, axis=1)
        
        # Concatenate agregated first level with the rest of the grid
        remaining_levels = ds_polluant.sel(lev=slice(ds_polluant.lev[17], ds_polluant.lev[-1]))
        ds_polluant_reduced = xr.concat([first_level, remaining_levels], dim="lev")
        
        # Update the lev dimension
        new_lev_values = [ds_polluant.lev[0].item()] + list(ds_polluant.lev[17:].values)
        ds_polluant_reduced = ds_polluant_reduced.assign_coords(lev=("lev", new_lev_values))


        # === Prepare the variables to compute the SRM ===
    
        # Create the new ariables using an empty dataarray with the corresponding dimensions
        new_var = xr.DataArray(
            data=np.zeros((ds_polluant_reduced.sizes["time"], ds_polluant_reduced.sizes["lev"]
                           , ds_polluant_reduced.sizes["lat"], ds_polluant_reduced.sizes["lon"])), 
            dims=("time", "lev", "lat", "lon"),
            coords={ "time": ds_polluant_reduced["time"], "lev": ds_polluant_reduced["lev"], 
                     "lat": ds_polluant_reduced["lat"],"lon": ds_polluant_reduced["lon"] }
        )
        
        ds_polluant_reduced["local_deposition"] = new_var
        ds_polluant_reduced["to_further_deposition"] = new_var
        ds_polluant_reduced["ratio_to_N"] = new_var
        ds_polluant_reduced["ratio_to_S"] = new_var
        ds_polluant_reduced["ratio_to_E"] = new_var
        ds_polluant_reduced["ratio_to_O"] = new_var
        ds_polluant_reduced["ratio_to_U"] = new_var
        ds_polluant_reduced["ratio_to_D"] = new_var
        
        # Load the initial (i_) diagnostics
        i_flux_in = ds_polluant_reduced["flux_in"].values
        i_flux_out = ds_polluant_reduced["flux_out"].values
        i_emission = ds_polluant_reduced["emission"].values
        i_deposition = ds_polluant_reduced["deposition"].values
        i_accumulation = ds_polluant_reduced["accumulation"].values
        i_reevaporation = ds_polluant_reduced["reevaporation"].values
        i_equilibrium = ds_polluant_reduced["equilibrium"].values        
        
        i_flux_out_N = ds_polluant_reduced["flux_out_N"].values
        i_flux_out_S = ds_polluant_reduced["flux_out_S"].values
        i_flux_out_E = ds_polluant_reduced["flux_out_E"].values
        i_flux_out_O = ds_polluant_reduced["flux_out_O"].values
        i_flux_out_U = ds_polluant_reduced["flux_out_U"].values
        i_flux_out_D = ds_polluant_reduced["flux_out_D"].values
        
        ntime, nlev, nlat, nlon = i_flux_in.shape

        # Compute local deposition
        in_i = i_flux_in + i_emission + i_reevaporation
        in_i = np.where(in_i == 0, np.nan, in_i)  # ou un petit epsilon
        local_deposition = (i_emission / in_i) * i_deposition

        # Compute transfers from local emissions
        to_further_deposition = (i_emission - local_deposition)
        
        # Compute ratios to each direction
        ratio_to_N = i_flux_out_N / i_flux_out
        ratio_to_S = i_flux_out_S / i_flux_out
        ratio_to_E = i_flux_out_E / i_flux_out
        ratio_to_O = i_flux_out_O / i_flux_out
        ratio_to_U = i_flux_out_U / i_flux_out
        ratio_to_D = i_flux_out_D / i_flux_out

        # === Save the file ===

        corrected_filename = f"prepared_{month_A}_{group_name}.nc"
        
        ds_polluant_reduced["local_deposition"].values = local_deposition        
        ds_polluant_reduced["to_further_deposition"].values = to_further_deposition
        
        ds_polluant_reduced["ratio_to_N"].values = ratio_to_N        
        ds_polluant_reduced["ratio_to_S"].values = ratio_to_S        
        ds_polluant_reduced["ratio_to_E"].values = ratio_to_E        
        ds_polluant_reduced["ratio_to_O"].values = ratio_to_O        
        ds_polluant_reduced["ratio_to_U"].values = ratio_to_U        
        ds_polluant_reduced["ratio_to_D"].values = ratio_to_D
                
        ds_polluant_reduced.to_netcdf(f"srm_computation_files/{corrected_filename}")        
        print(f"Saved : {corrected_filename}")

Fichier préparé enregistré : monthly/201901/NE_prepared_01_nhx.nc
Fichier préparé enregistré : monthly/201901/NE_prepared_01_nox.nc
Fichier préparé enregistré : monthly/201901/NE_prepared_01_sox.nc
Fichier préparé enregistré : monthly/201902/NE_prepared_02_nhx.nc
Fichier préparé enregistré : monthly/201902/NE_prepared_02_nox.nc
Fichier préparé enregistré : monthly/201902/NE_prepared_02_sox.nc
Fichier préparé enregistré : monthly/201903/NE_prepared_03_nhx.nc
Fichier préparé enregistré : monthly/201903/NE_prepared_03_nox.nc
Fichier préparé enregistré : monthly/201903/NE_prepared_03_sox.nc
Fichier préparé enregistré : monthly/201904/NE_prepared_04_nhx.nc
Fichier préparé enregistré : monthly/201904/NE_prepared_04_nox.nc
Fichier préparé enregistré : monthly/201904/NE_prepared_04_sox.nc
Fichier préparé enregistré : monthly/201905/NE_prepared_05_nhx.nc
Fichier préparé enregistré : monthly/201905/NE_prepared_05_nox.nc
Fichier préparé enregistré : monthly/201905/NE_prepared_05_sox.nc
Fichier pr

In [28]:
# SRM computation

# Create folder to store the SRM files
dossier = "SRM_files"
os.makedirs(dossier, exist_ok=True)

for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        # Load file
        filename = f"prepared_{month_A}_{group_name}"
        ds_prepared = xr.open_dataset(f"srm_computation_files/{filename}.nc")
        
        # === Initialize the SRM ===

        # Create the grid and inverse the dimensions with indexing ij
        latitudes = [-89.5, -88. , -86. , -84. , -82. , -80. , -78. , -76. , -74. , -72. ,
                    -70. , -68. , -66. , -64. , -62. , -60. , -58. , -56. , -54. , -52. ,
                    -50. , -48. , -46. , -44. , -42. , -40. , -38. , -36. , -34. , -32. ,
                    -30. , -28. , -26. , -24. , -22. , -20. , -18. , -16. , -14. , -12. ,
                    -10. ,  -8. ,  -6. ,  -4. ,  -2. ,   0. ,   2. ,   4. ,   6. ,   8. ,
                    10. ,  12. ,  14. ,  16. ,  18. ,  20. ,  22. ,  24. ,  26. ,  28. ,
                    30. ,  32. ,  34. ,  36. ,  38. ,  40. ,  42. ,  44. ,  46. ,  48. ,
                    50. ,  52. ,  54. ,  56. ,  58. ,  60. ,  62. ,  64. ,  66. ,  68. ,
                    70. ,  72. ,  74. ,  76. ,  78. ,  80. ,  82. ,  84. ,  86. ,  88. ,
                    89.5]
        longitudes = [-180. , -177.5, -175. , -172.5, -170. , -167.5, -165. , -162.5, -160. ,
                    -157.5, -155. , -152.5, -150. , -147.5, -145. , -142.5, -140. , -137.5,
                    -135. , -132.5, -130. , -127.5, -125. , -122.5, -120. , -117.5, -115. ,
                    -112.5, -110. , -107.5, -105. , -102.5, -100. ,  -97.5,  -95. ,  -92.5,
                    -90. ,  -87.5,  -85. ,  -82.5,  -80. ,  -77.5,  -75. ,  -72.5,  -70. ,
                    -67.5,  -65. ,  -62.5,  -60. ,  -57.5,  -55. ,  -52.5,  -50. ,  -47.5,
                    -45. ,  -42.5,  -40. ,  -37.5,  -35. ,  -32.5,  -30. ,  -27.5,  -25. ,
                    -22.5,  -20. ,  -17.5,  -15. ,  -12.5,  -10. ,   -7.5,   -5. ,   -2.5,
                    0. ,    2.5,    5. ,    7.5,   10. ,   12.5,   15. ,   17.5,   20. ,
                    22.5,   25. ,   27.5,   30. ,   32.5,   35. ,   37.5,   40. ,   42.5,
                    45. ,   47.5,   50. ,   52.5,   55. ,   57.5,   60. ,   62.5,   65. ,
                    67.5,   70. ,   72.5,   75. ,   77.5,   80. ,   82.5,   85. ,   87.5,
                    90. ,   92.5,   95. ,   97.5,  100. ,  102.5,  105. ,  107.5,  110. ,
                    112.5,  115. ,  117.5,  120. ,  122.5,  125. ,  127.5,  130. ,  132.5,
                    135. ,  137.5,  140. ,  142.5,   145. ,  147.5,  150. ,  152.5,  155. ,
                    157.5,  160. ,  162.5,  165. ,  167.5,  170. ,  172.5,  175. ,  177.5]
        
        lat_grid, lon_grid = np.meshgrid(latitudes, longitudes, indexing='ij')
        
        # Flatten and stack coordinates
        lat_lon_vector = np.vstack((lat_grid.flatten(), lon_grid.flatten())).T
        
        # Initialize list to store column names
        column_names = ["lat", "lon"]
        
        # Pre-extract all arrays 
        i_flux_in = ds_prepared["flux_in"].values
        i_flux_out = ds_prepared["flux_out"].values
        i_emission = ds_prepared["emission"].values
        i_deposition = ds_prepared["deposition"].values
        i_accumulation = ds_prepared["accumulation"].values
        i_reevaporation = ds_prepared["reevaporation"].values
        i_equilibrium = ds_prepared["equilibrium"].values
        
        i_ratio_to_N = ds_prepared["ratio_to_N"].values
        i_ratio_to_S = ds_prepared["ratio_to_S"].values
        i_ratio_to_E = ds_prepared["ratio_to_E"].values
        i_ratio_to_O = ds_prepared["ratio_to_O"].values
        i_ratio_to_U = ds_prepared["ratio_to_U"].values
        i_ratio_to_D = ds_prepared["ratio_to_D"].values
        
        i_local_deposition = ds_prepared["local_deposition"].values
        i_to_further_deposition = ds_prepared["to_further_deposition"].values
        
        ntime, nlev, nlat, nlon = i_to_further_deposition.shape

        # === Process transport for a single source level with its specific threshold ===
        
        def process_level_transport(lev_cc, lat_cc, lon_cc, source_threshold):
            
            # Initialize transport and deposition arrays for this level
            transport = np.zeros_like(i_local_deposition)
            loc_dep = np.zeros_like(i_local_deposition)
            
            # Get initial transport from this specific level
            to_transport_i = i_to_further_deposition[0, lev_cc, lat_cc, lon_cc]
            
            if to_transport_i == 0:
                return np.zeros_like(i_local_deposition)
            
            # Pre-extract ratios for initial transport
            ratio_to_N = i_ratio_to_N[0, lev_cc, lat_cc, lon_cc]
            ratio_to_S = i_ratio_to_S[0, lev_cc, lat_cc, lon_cc]
            ratio_to_E = i_ratio_to_E[0, lev_cc, lat_cc, lon_cc]
            ratio_to_O = i_ratio_to_O[0, lev_cc, lat_cc, lon_cc]
            ratio_to_U = i_ratio_to_U[0, lev_cc, lat_cc, lon_cc]
            ratio_to_D = i_ratio_to_D[0, lev_cc, lat_cc, lon_cc]
            
            # Calculate initial transports
            transport_N = to_transport_i * ratio_to_N
            transport_S = to_transport_i * ratio_to_S
            transport_E = to_transport_i * ratio_to_E
            transport_O = to_transport_i * ratio_to_O
            transport_U = to_transport_i * ratio_to_U
            transport_D = to_transport_i * ratio_to_D
        
            # Initial neighbor updates
            lat_n = lat_cc + 1 if lat_cc + 1 < nlat else None
            lat_s = lat_cc - 1 if lat_cc - 1 >= 0 else None
            lon_e = (lon_cc + 1) % nlon
            lon_w = (lon_cc - 1) % nlon
            lev_u = lev_cc - 1 if lev_cc - 1 >= 0 else None 
            lev_d = lev_cc + 1 if lev_cc + 1 < nlev else None
        
            if lat_n is not None:
                transport[0, lev_cc, lat_n, lon_cc] += transport_N
            if lat_s is not None:
                transport[0, lev_cc, lat_s, lon_cc] += transport_S
            transport[0, lev_cc, lat_cc, lon_e] += transport_E
            transport[0, lev_cc, lat_cc, lon_w] += transport_O
            if lev_u is not None:
                transport[0, lev_u, lat_cc, lon_cc] += transport_U
            if lev_d is not None:
                transport[0, lev_d, lat_cc, lon_cc] += transport_D
            
            # Iterative transport with source-specific threshold
            iteration = 0
            max_iterations = 1000
            
            while np.any(transport > source_threshold) and iteration < max_iterations:
                # Create mask based on source threshold
                mask = transport > source_threshold
                
                if not np.any(mask):
                    break
                    
                # Get indices where transport exceeds threshold
                indices = np.where(mask)
                
                if len(indices[0]) == 0:
                    break
                
                # Extract relevant values at mask positions
                lev_indices, lat_indices, lon_indices = indices[1], indices[2], indices[3]
                
                # Pre-compute values for valid indices
                ratios_N = i_ratio_to_N[0, lev_indices, lat_indices, lon_indices]
                ratios_S = i_ratio_to_S[0, lev_indices, lat_indices, lon_indices]
                ratios_E = i_ratio_to_E[0, lev_indices, lat_indices, lon_indices]
                ratios_O = i_ratio_to_O[0, lev_indices, lat_indices, lon_indices]
                ratios_U = i_ratio_to_U[0, lev_indices, lat_indices, lon_indices]
                ratios_D = i_ratio_to_D[0, lev_indices, lat_indices, lon_indices]
                
                emissions = i_emission[0, lev_indices, lat_indices, lon_indices]
                flux_ins = i_flux_in[0, lev_indices, lat_indices, lon_indices]
                reevaps = i_reevaporation[0, lev_indices, lat_indices, lon_indices]
                depositions = i_deposition[0, lev_indices, lat_indices, lon_indices]
                
                transport_vals = transport[mask]
                
                # Calculate local deposition
                in_j = flux_ins + emissions + reevaps
                dep_loc_jj = np.where(
                    in_j != 0,
                    depositions * (transport_vals / in_j),
                    0
                )
                
                to_transport_j = transport_vals - dep_loc_jj
                
                # Update local deposition at mask positions
                loc_dep[mask] += dep_loc_jj
                
                # Calculate new transports
                transport_N = to_transport_j * ratios_N
                transport_S = to_transport_j * ratios_S
                transport_E = to_transport_j * ratios_E
                transport_O = to_transport_j * ratios_O
                transport_U = to_transport_j * ratios_U
                transport_D = to_transport_j * ratios_D
                
                # Reset transport array
                transport = np.zeros_like(transport)
                
                # Vectorized neighbor calculations
                lat_n = np.clip(lat_indices + 1, 0, nlat - 1)
                lat_s = np.clip(lat_indices - 1, 0, nlat - 1)
                lon_e = (lon_indices + 1) % nlon
                lon_w = (lon_indices - 1) % nlon
                lev_u = np.clip(lev_indices - 1, 0, nlev - 1)
                lev_d = np.clip(lev_indices + 1, 0, nlev - 1)
                
                # Boundary checks
                valid_n = lat_indices + 1 < nlat
                valid_s = lat_indices - 1 >= 0
                valid_u = lev_indices - 1 >= 0
                valid_d = lev_indices + 1 < nlev
                
                # Add transports to new positions
                if np.any(valid_n):
                    np.add.at(transport, (0, lev_indices[valid_n], lat_n[valid_n], lon_indices[valid_n]), 
                             transport_N[valid_n])
                if np.any(valid_s):
                    np.add.at(transport, (0, lev_indices[valid_s], lat_s[valid_s], lon_indices[valid_s]), 
                             transport_S[valid_s])
                
                np.add.at(transport, (0, lev_indices, lat_indices, lon_e), transport_E)
                np.add.at(transport, (0, lev_indices, lat_indices, lon_w), transport_O)
                
                if np.any(valid_u):
                    np.add.at(transport, (0, lev_u[valid_u], lat_indices[valid_u], lon_indices[valid_u]), 
                             transport_U[valid_u])
                if np.any(valid_d):
                    np.add.at(transport, (0, lev_d[valid_d], lat_indices[valid_d], lon_indices[valid_d]), 
                             transport_D[valid_d])
                
                iteration += 1
                
                # Optionally reduce threshold to speed convergence
                #if iteration % 100 == 0:
                #    source_threshold *= 0.9
            
            return loc_dep
        
        # === SRM processing loop ===
        
        for lat_cc in tqdm(range(nlat), desc="Latitudes"):
            for lon_cc in range(nlon):
                
                # Calculate total emission from this source cell
                emis_src = np.sum(i_emission[0, :nlev, lat_cc, lon_cc])
                
                if emis_src == 0:
                    # Add empty column for zero-emission sources
                    vecteur_1D = np.zeros(nlat * nlon)
                    lat_lon_vector = np.column_stack((lat_lon_vector, vecteur_1D))
                    column_names.append(f"{latitudes[lat_cc]}_{longitudes[lon_cc]}")
                    continue
                
                # Initialize total deposition array for this source
                total_loc_dep = np.zeros_like(i_local_deposition)
                
                # Process each level separately with its own threshold
                for lev_cc in range(nlev):
                    emis_level = i_emission[0, lev_cc, lat_cc, lon_cc]
                    
                    if emis_level > 0:  # Only process levels with emissions
                        # Calculate threshold based on THIS source level's emission
                        source_threshold = 0.001 * emis_level
                        
                        # Process transport for this level with its specific threshold
                        level_deposition = process_level_transport(lev_cc, lat_cc, lon_cc, source_threshold)
                        
                        # Add to total deposition
                        total_loc_dep += level_deposition
                
                # Final calculations
                loc_dep_2D = np.sum(total_loc_dep, axis=1)  # Sum over levels
                
                if emis_src != 0:
                    loc_dep_2D = loc_dep_2D / emis_src  # Normalize by total source emission
                    
                variable_2D = loc_dep_2D[0]
                vecteur_1D = variable_2D.flatten()
                lat_lon_vector = np.column_stack((lat_lon_vector, vecteur_1D))
        
                # Add column name with lat/lon coordinates of the emitting element
                column_names.append(f"{latitudes[lat_cc]}_{longitudes[lon_cc]}")
        
        print(lat_lon_vector.shape)
        
        # Convert to DataFrame and save
        df = pd.DataFrame(lat_lon_vector, columns=column_names)
        df.to_csv(f"SRM_files/SRM_raw_{month_A}_{group_name}.csv", index=False)

Latitudes: 100%|█████████████████████████████| 91/91 [2:49:23<00:00, 111.68s/it]


(13104, 13106)


Latitudes: 100%|█████████████████████████████| 91/91 [7:04:26<00:00, 279.85s/it]


(13104, 13106)


Latitudes: 100%|█████████████████████████████| 91/91 [4:35:15<00:00, 181.49s/it]


(13104, 13106)


... (62 outputs left out)

Latitudes: 100%|█████████████████████████████| 91/91 [6:04:29<00:00, 240.33s/it]


(13104, 13106)


Latitudes: 100%|█████████████████████████████| 91/91 [4:16:01<00:00, 168.81s/it]


(13104, 13106)


### SRM Refinement

In [5]:
# Functions to load and save SRM files faster
def load_srm_matrix_ultra_fast(csv_file):
    
    df = pd.read_csv(csv_file)
    
    # Extract receptor coordinates
    lat_r_vals = np.sort(df['lat'].unique())
    lon_r_vals = np.sort(df['lon'].unique())
    
    # Extract source coordinates from column names
    source_cols = [col for col in df.columns if col not in ['lat', 'lon']]
    
    # More efficient coordinate extraction
    lat_s_set = set()
    lon_s_set = set()
    source_coord_dict = {}
    
    for col in source_cols:
        lat_s, lon_s = map(float, col.split('_'))
        lat_s_set.add(lat_s)
        lon_s_set.add(lon_s)
        source_coord_dict[col] = (lat_s, lon_s)
    
    lat_s_vals = np.sort(list(lat_s_set))
    lon_s_vals = np.sort(list(lon_s_set))
    
    # Create coordinate mappings using searchsorted for efficiency
    lat_r_indices = np.searchsorted(lat_r_vals, df['lat'])
    lon_r_indices = np.searchsorted(lon_r_vals, df['lon'])
    
    lat_s_idx_map = {val: i for i, val in enumerate(lat_s_vals)}
    lon_s_idx_map = {val: i for i, val in enumerate(lon_s_vals)}
    
    # Initialize SRM matrix
    srm = np.full((len(lat_r_vals), len(lon_r_vals), len(lat_s_vals), len(lon_s_vals)), np.nan)

    # Ultra-fast filling using advanced indexing
    for col in tqdm(source_cols, desc="Processing columns"):
        lat_s, lon_s = source_coord_dict[col]
        isrc = lat_s_idx_map[lat_s]
        jsrc = lon_s_idx_map[lon_s]
        
        # Single vectorized assignment
        srm[lat_r_indices, lon_r_indices, isrc, jsrc] = df[col].values
    
    return srm, lat_r_vals, lon_r_vals, lat_s_vals, lon_s_vals

def load_srm_matrix_reshape_method(csv_file):
    
    df = pd.read_csv(csv_file)
    
    # Extract coordinates
    lat_r_vals = np.sort(df['lat'].unique())
    lon_r_vals = np.sort(df['lon'].unique())
    source_cols = [col for col in df.columns if col not in ['lat', 'lon']]
    
    # Parse source coordinates
    source_coords = np.array([list(map(float, col.split('_'))) for col in source_cols])
    lat_s_vals = np.sort(np.unique(source_coords[:, 0]))
    lon_s_vals = np.sort(np.unique(source_coords[:, 1]))
    
    print(f"Dimensions: {len(lat_r_vals)} x {len(lon_r_vals)} x {len(lat_s_vals)} x {len(lon_s_vals)}")
    
    # Check if data is on a regular grid (can use faster reshape method)
    expected_rows = len(lat_r_vals) * len(lon_r_vals)
    if len(df) == expected_rows:
        print("Regular grid detected - using ultra-fast reshape method")
        
        # Sort dataframe by coordinates
        df_sorted = df.sort_values(['lat', 'lon']).reset_index(drop=True)
        
        # Initialize SRM matrix
        srm = np.full((len(lat_r_vals), len(lon_r_vals), len(lat_s_vals), len(lon_s_vals)), np.nan)
        
        # Create source coordinate mappings
        lat_s_idx_map = {val: i for i, val in enumerate(lat_s_vals)}
        lon_s_idx_map = {val: i for i, val in enumerate(lon_s_vals)}
        
        # Ultra-fast filling using reshape
        for col in tqdm(source_cols, desc="Processing columns"):
            lat_s, lon_s = map(float, col.split('_'))
            isrc = lat_s_idx_map[lat_s]
            jsrc = lon_s_idx_map[lon_s]
            
            # Reshape the column data directly into the 2D receptor grid
            data_2d = df_sorted[col].values.reshape(len(lat_r_vals), len(lon_r_vals))
            srm[:, :, isrc, jsrc] = data_2d
        
        return srm, lat_r_vals, lon_r_vals, lat_s_vals, lon_s_vals
    
    else:
        print("Irregular grid detected - falling back to standard method")
        return load_srm_matrix_ultra_fast(csv_file)

def save_srm_matrix_to_csv_fast(srm_matrix, lat_r_vals, lon_r_vals, lat_s_vals, lon_s_vals, output_file):
    
    nlat_r, nlon_r, nlat_s, nlon_s = srm_matrix.shape

    # Flatten 4D to 2D - each row = one receptor, each column = one source
    mat2d = srm_matrix.reshape(nlat_r * nlon_r, nlat_s * nlon_s)

    # Create columns with receptor coordinates
    lat_r_grid, lon_r_grid = np.meshgrid(lat_r_vals, lon_r_vals, indexing="ij")
    lat_r_flat = lat_r_grid.ravel()
    lon_r_flat = lon_r_grid.ravel()

    # Name columns with source element
    source_cols = [f"{lat_s}_{lon_s}" for lat_s in lat_s_vals for lon_s in lon_s_vals]

    # Build DataFrame
    df_out = pd.DataFrame(mat2d, columns=source_cols)
    df_out.insert(0, "lon", lon_r_flat)  # mettre lon juste après lat
    df_out.insert(0, "lat", lat_r_flat)

    # Save
    df_out.to_csv(output_file, index=False)
    print(f"✅ SRM matrix saved to {output_file} (shape {df_out.shape})")


In [6]:
# SRM Refinement 1 & 2

# Load mask to identify land and sea area
mask_terremer = np.loadtxt("ext_input/land0_sea1.csv", delimiter=",", dtype=int)

for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        
        # === First refinement ===
        
        ds_prepared = xr.open_dataset(f'srm_computation_files/prepared_{month_A}_{group_name}.nc')
        i_deposition = ds_prepared["deposition"].values
        i_emission = ds_prepared["emission"].values

        ntime, nlev, nlat, nlon  = i_deposition.shape

        srm, lat_r_vals, lon_r_vals, lat_s_vals, lon_s_vals = load_srm_matrix_reshape_method(f"SRM_files/SRM_raw_{month_A}_{group_name}.csv")
        
        # Select value at time = 0 (lev, lat, lon)
        dep_inter = i_deposition[0]
        emis_inter = i_emission[0]
        
        # Sum on lev dimension (axis 0)
        dep_2D = dep_inter.sum(axis=0)
        emis_2D = emis_inter.sum(axis=0)

        missing = np.zeros_like(dep_2D)

        # Calculate the missing deposition (due to threshold 0.001%)
        for lat_r in tqdm(range(nlat), desc="Latitudes"):
            for lon_r in range(nlon):
                
                tot_dep = dep_2D[lat_r, lon_r]
                missing_amount = 0
                
                for lat_s in range(nlat):
                    for lon_s in range(nlon):
                        missing_amount += srm[lat_r, lon_r, lat_s, lon_s] * emis_2D[lat_s, lon_s]
        
                        
                # missing_r = dep_r - SUM_s(srm_s_r * emis_s)
                missing[lat_r, lon_r] = tot_dep - missing_amount
                if missing[lat_r, lon_r] <0:
                    missing[lat_r, lon_r]=0

        # Calculate the available fraction of emissions
        available = np.zeros_like(emis_2D)
        for lat_s in tqdm(range(nlat), desc="Latitudes"):
            for lon_s in range(nlon):
                
                emis_s = emis_2D[lat_s, lon_s]
                rest_available = 0
                
                for lat_r in range(nlat):
                    for lon_r in range(nlon):
                        rest_available += srm[lat_r, lon_r, lat_s, lon_s] * emis_s
        
                        
                # available_s = emis_s - SUM_r(srm_s_r * emis_s)        
                available[lat_s, lon_s] = emis_s - rest_available
                if available[lat_s, lon_s] <0:
                    available[lat_s, lon_s]=0

        srm_prime = srm.copy()

        # Add the missing or available fraction to the srm
        for lat_r in tqdm(range(nlat), desc="Latitudes"):
            for lon_r in range(nlon):
                
                missing_r = missing[lat_r, lon_r]
                
                for lat_s in range(nlat):
                    for lon_s in range(nlon):
                        atmosphere_s = available[lat_s, lon_s] * srm[lat_r, lon_r, lat_s, lon_s]

                        if missing_r < atmosphere_s and atmosphere_s != 0 :
                            add = missing_r * (missing_r/atmosphere_s)
                            srm_prime[lat_r, lon_r, lat_s, lon_s] += add
                        elif atmosphere_s < missing_r :
                            srm_prime[lat_r, lon_r, lat_s, lon_s] += atmosphere_s

        missing_check = np.zeros_like(dep_2D)

        # Check the equality between total deposition and srm_prime*emission
        for lat_r in tqdm(range(nlat), desc="Latitudes"):
            for lon_r in range(nlon):
                
                tot_dep = dep_2D[lat_r, lon_r]
                missing_amount = 0
                
                for lat_s in range(nlat):
                    for lon_s in range(nlon):
                        missing_amount += srm_prime[lat_r, lon_r, lat_s, lon_s] * emis_2D[lat_s, lon_s]
        
                        
                # missing_r = dep_r - SUM_s(srm_s_r * emis_s)
                missing_check[lat_r, lon_r] = tot_dep - missing_amount

        if np.sum(np.abs(missing_check) < 1e-20) != 0:
            print(f"Check missing correction for {month_A}_{group_name}")

        # === Second refinement ===

        col_sums_prime = np.zeros_like(emis_2D)

        # Get 2D SRM
        for lat_s in range(nlat) :
            for lon_s in range(nlon):
                
                column_sum = srm_prime[ :, :, lat_s, lon_s].sum()
                col_sums_prime[lat_s, lon_s] = column_sum
        
        srm_second = srm_prime.copy()

        # Apply correction depending on the pollutant
        if group_name != "nox":
            if group_name == "nhx":
                # Charger les données du fichier
                file_A = xr.open_dataset(f'geos_diags_converted/geos_diags_converted_2019{month_A}.nc')
                file_prec = xr.open_dataset(f'ext_input/meteo/meteo_2019{month_A}.SUB.nc')
    
                i_prectot = file_prec["PRECTOT"].values
                i_drydepvel = file_A["DryDepVel_NH3"].values
                
                prectot = i_prectot[0] # mm/s
                drydepvel = i_drydepvel[0] # molec/cm²/s
    
                prectot_cm = prectot / 10
                loc_dep_frac = (0.19 + (0.19*prectot_cm/(prectot_cm+(drydepvel/1.4e6))))
    
            if group_name == "sox":
                loc_dep_frac = np.where(mask_terremer == 0, 0.15, 0.1)
    
            for lat_s in tqdm(range(nlat), desc="Latitudes"):
                for lon_s in range(nlon):                
                    
                    srm_sum = col_sums_prime[lat_s, lon_s]
                    source_dep_frac = loc_dep_frac[lat_s, lon_s]
                    
                    if srm_sum != 0:
                        emis_s = emis_2D[lat_s, lon_s]
                        for lat_r in range(nlat):
                            for lon_r in range(nlon):
                                
                                if lat_r == lat_s and lon_r == lon_s:                        
                                    srm_second[lat_r, lon_r, lat_s, lon_s] += source_dep_frac
                                    
                                else:
                                    ratio = srm_prime[lat_r, lon_r, lat_s, lon_s] / srm_sum
                                    recep_dep_frac = ratio * source_dep_frac
                                    if recep_dep_frac < srm_second[lat_r, lon_r, lat_s, lon_s]:
                                        srm_second[lat_r, lon_r, lat_s, lon_s] -= recep_dep_frac
        
        elif group_name == "nox":
            
            for lat_s in tqdm(range(nlat), desc="Latitudes"):
                for lon_s in range(nlon):                
                    
                    srm_sum = col_sums_prime[lat_s, lon_s]
            
                    if srm_sum != 0:
                        emis_s = emis_2D[lat_s, lon_s]
                        for lat_r in range(nlat):
                            for lon_r in range(nlon):
                                
                                if lat_r == lat_s and lon_r == lon_s:                        
                                    srm_second[lat_r, lon_r, lat_s, lon_s] += 0.04
                                    
                                else:
                                    ratio = srm_prime[lat_r, lon_r, lat_s, lon_s] / srm_sum
                                    recep_dep_frac = 0.04 * ratio
                                    if recep_dep_frac < srm_second[lat_r, lon_r, lat_s, lon_s]:
                                        srm_second[lat_r, lon_r, lat_s, lon_s] -= recep_dep_frac
        
        output_csv = f"SRM_files/SRM_refined_{month_A}_{group_name}.csv"
        save_srm_matrix_to_csv_fast(srm_second, lat_r_vals, lon_r_vals, lat_s_vals, lon_s_vals, output_csv)

Loading CSV file...
Dimensions: 91 x 144 x 91 x 144
Regular grid detected - using ultra-fast reshape method


Latitudes: 100%|████████████████████████████████| 91/91 [01:39<00:00,  1.10s/it]


✅ SRM matrix saved to monthly/201912/NE_SRM_second_12_nhx.csv (shape (13104, 13106))
Loading CSV file...
Dimensions: 91 x 144 x 91 x 144
Regular grid detected - using ultra-fast reshape method


Latitudes: 100%|████████████████████████████████| 91/91 [02:04<00:00,  1.37s/it]


✅ SRM matrix saved to monthly/201912/NE_SRM_second_12_nox.csv (shape (13104, 13106))
Loading CSV file...
Dimensions: 91 x 144 x 91 x 144
Regular grid detected - using ultra-fast reshape method


Latitudes: 100%|████████████████████████████████| 91/91 [01:45<00:00,  1.16s/it]


✅ SRM matrix saved to monthly/201912/NE_SRM_second_12_sox.csv (shape (13104, 13106))


### Display SRM results for a single source element

In [212]:
# Display the SRM results for a single cell - preparation

# Source cell column name = lat_lon
source_cell = "58.0_-70.0"

# Extract lat/lon from source cell
lat_str, lon_str = third_col_name.split("_")
lat_val = float(lat_str)
lon_val = float(lon_str)

# Lists of lon and lat + conversion to np arrays
lon_list = [180., -177.5, -175., -172.5, -170., -167.5, -165., -162.5, -160.,
       -157.5, -155., -152.5, -150., -147.5, -145., -142.5, -140., -137.5,
       -135., -132.5, -130., -127.5, -125., -122.5, -120., -117.5, -115.,
       -112.5, -110., -107.5, -105., -102.5, -100., -97.5, -95., -92.5,
       -90., -87.5, -85., -82.5, -80., -77.5, -75., -72.5, -70.,
       -67.5, -65., -62.5, -60., -57.5, -55., -52.5, -50., -47.5,
       -45., -42.5, -40., -37.5, -35., -32.5, -30., -27.5, -25.,
       -22.5, -20., -17.5, -15., -12.5, -10., -7.5, -5., -2.5,
        0., 2.5, 5., 7.5, 10., 12.5, 15., 17.5, 20.,
        22.5, 25., 27.5, 30., 32.5, 35., 37.5, 40., 42.5,
        45., 47.5, 50., 52.5, 55., 57.5, 60., 62.5, 65.,
        67.5, 70., 72.5, 75., 77.5, 80., 82.5, 85., 87.5,
        90., 92.5, 95., 97.5, 100., 102.5, 105., 107.5, 110.,
        112.5, 115., 117.5, 120., 122.5, 125., 127.5, 130., 132.5,
        135., 137.5, 140., 142.5, 145., 147.5, 150., 152.5, 155.,
        157.5, 160., 162.5, 165., 167.5, 170., 172.5, 175., 177.5]
lat_list = [-89.5, -88., -86., -84., -82., -80., -78., -76., -74., -72.,
       -70., -68., -66., -64., -62., -60., -58., -56., -54., -52.,
       -50., -48., -46., -44., -42., -40., -38., -36., -34., -32.,
       -30., -28., -26., -24., -22., -20., -18., -16., -14., -12.,
       -10., -8., -6., -4., -2., 0., 2., 4., 6., 8.,
        10., 12., 14., 16., 18., 20., 22., 24., 26., 28.,
        30., 32., 34., 36., 38., 40., 42., 44., 46., 48.,
        50., 52., 54., 56., 58., 60., 62., 64., 66., 68.,
        70., 72., 74., 76., 78., 80., 82., 84., 86., 88.,
        89.5]

lat = np.array(lat_list)
lon = np.array(lon_list)



In [196]:
# Display the SRM results for a single cell - load data & display

# Load file
for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():

        filename = f"SRM_refined_{month_A}_{group_name}"
        
        df = pd.read_csv(f"SRM_files/{filename}.csv")
        
        # filter non-null values
        df_filtered = df[df[source_cell] != 0]
        
        # Extract filtered coordinates
        lat_inds = df_filtered['lat'].values
        lon_inds = df_filtered['lon'].values
        
        # Generate the map
        plt.figure(figsize=(14, 7))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_global()
        
        # Generate the grid
        for la in lat:
            ax.plot(lon, [la] * len(lon), color='lightgray', linewidth=0.5, transform=ccrs.PlateCarree())
        for lo in lon:
            ax.plot([lo] * len(lat), lat, color='lightgray', linewidth=0.5, transform=ccrs.PlateCarree())
        
        # Get filtered values for each coordinate
        valeurs = df_filtered[source_cell].values
        
        # Filter positive values to avoid errors
        valeurs_pos = valeurs[valeurs > 0]
        lon_inds_pos = lon_inds[valeurs > 0]
        lat_inds_pos = lat_inds[valeurs > 0]
        
        # Log Normalization
        norm = mcolors.LogNorm(vmin=valeurs_pos.min(), vmax=valeurs_pos.max())
        
        # Colormap
        cmap = plt.cm.Reds
        
        # Define color based on normalized values
        sc = ax.scatter(
            lon_inds_pos, lat_inds_pos,
            c=valeurs_pos,
            cmap=cmap,
            norm=norm,
            s=20,
            transform=ccrs.PlateCarree(),
            label="Emission to deposition ratio"
        )
        
        
        ax.scatter(lon_val, lat_val, color='blue', s=10, transform=ccrs.PlateCarree(), label="Emitting cell")
        
        # Add map elements
        ax.coastlines(resolution='110m', color='black')
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
        
        # Display or save
        plt.title(filename)
        plt.tight_layout()
        plt.savefig(f"{filename}.png", dpi=300)
        plt.close()
        #plt.show()

### Cell CF calculation

In [7]:
# Calculation for terrestrial acidification

dossier = "CF_files"
os.makedirs(dossier, exist_ok=True)

# Load Response Factors
rf_0 = pd.read_csv("cell_RF_TERRAC.csv")
rf_0["lat"] = rf_0["lat"].astype(float)
rf_0["lon"] = rf_0["lon"].astype(float)

# Main loop
for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        # Load SRM
        srm = pd.read_csv(f"SRM_files/SRM_refined_{month_A}_{group_name}.csv")
        
        # Set index by lat/lon
        rf = rf_0.set_index(["lat", "lon"])
        
        # List to stock results
        results = []
        
        # Loop on each column = emitter (except lat, lon)
        for col in srm.columns[2:]:
            # Extract emitter coordinates from column name
            lat_c, lon_c = map(float, col.split("_"))
            
            # Calculation RF * SRM for each row
            merged = srm[["lat", "lon", col]].merge(rf, left_on=["lat", "lon"], right_index=True)
            cf_val = (merged[col] * merged[f"RF_{group_name}"]).sum()
            
            # Stock result
            results.append((lat_c, lon_c, cf_val))
        
        # Transform to DataFrame
        df_final = pd.DataFrame(results, columns=["lat", "lon", f"CF_{group_name}"])
        
        # Save CSV
        df_final.to_csv(f"CF_files/CF_TERRAC_{month_A}_{group_name}.csv", index=False)
        print(f"✅ File CF_TERRAC_{month_A}_{group_name}.csv created :", df_final.shape)


✅ Fichier NE6D_CF_TERRAC_01_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_01_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_01_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_02_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_02_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_02_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_03_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_03_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_03_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_04_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_04_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_04_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_05_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_05_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_05_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_06_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_06_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_06_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_TERRAC_07_

In [11]:
# Calculation for marine eutrophication

dossier = "CF_files"
os.makedirs(dossier, exist_ok=True)

# Load Response Factors
ca_0 = pd.read_csv("cell_RF_MAREUT.csv")
ca_0["lat"] = ca_0["lat"].astype(float)
ca_0["lon"] = ca_0["lon"].astype(float)

# Main loop
for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        # Load SRM
        srm = pd.read_csv(f"SRM_files/SRM_refined_{month_A}_{group_name}.csv")
        
        # Set index by lat/lon
        ca = ca_0.set_index(["lat", "lon"])
        
        # List to stock results
        results = []
        
        # Loop on each column = emitter (except lat, lon)
        for col in srm.columns[2:]:
            # Extract emitter coordinates from column name
            lat_c, lon_c = map(float, col.split("_"))
            
            # Calculation CA * SRM for each row
            merged = srm[["lat", "lon", col]].merge(ca, left_on=["lat", "lon"], right_index=True)
            cf_val = (merged[col] * merged["CA"]).sum()
            
            # Stock result
            results.append((lat_c, lon_c, cf_val))
        
        # Transform to DataFrame
        df_final = pd.DataFrame(results, columns=["lat", "lon", f"CF_{group_name}"])
        
        # Save CSV
        df_final.to_csv(f"CF_files/CF_MAREUT_{month_A}_{group_name}.csv", index=False)
        print(f"✅ File CF_MAREUT_{month_A}_{group_name}.csv created :", df_final.shape)


✅ Fichier NE_CF_EUTROMAR_01_nhx.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_01_nox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_01_sox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_02_nhx.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_02_nox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_02_sox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_03_nhx.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_03_nox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_03_sox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_04_nhx.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_04_nox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_04_sox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_05_nhx.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_05_nox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_05_sox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_06_nhx.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_06_nox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_06_sox.csv créé : (13104, 3)
✅ Fichier NE_CF_EUTROMAR_07_

In [8]:
# Calculation for freshwater acidification

dossier = "CF_files"
os.makedirs(dossier, exist_ok=True)

# Load Response Factors
rf_0 = pd.read_csv("cell_RF_FWAC.csv")
rf_0["lat"] = rf_0["lat"].astype(float)
rf_0["lon"] = rf_0["lon"].astype(float)

# Main loop
for month_A, month_B in next_month.items():
    for group_name, chems in sub_groups.items():
        # Load SRM
        srm = pd.read_csv(f"SRM_files/SRM_refined_{month_A}_{group_name}.csv")
        
        # Set index by lat/lon
        rf = rf_0.set_index(["lat", "lon"])
        
        # List to stock results
        results = []

        # Factors of kg to keq for each pollutant
        kg_to_keq = 1
        if group_name == "nhx":
            kg_to_keq = 8/18
        elif group_name == "nox":
            kg_to_keq = 24/62
        else:
            kg_to_keq = 32/96
        
        # Loop on each column = emitter (except lat, lon)
        for col in srm.columns[2:]:
            # Extract emitter coordinates from column name
            lat_c, lon_c = map(float, col.split("_"))
            
            # Calculation RF * kg_to_keq * SRM for each row
            merged = srm[["lat", "lon", col]].merge(rf, left_on=["lat", "lon"], right_index=True)
            cf_val = (merged[col] * kg_to_keq * merged[f"RF_{group_name}_{month_A}"]).sum()
            
            # Stock result
            results.append((lat_c, lon_c, cf_val))
        
        # Transform to DataFrame
        df_final = pd.DataFrame(results, columns=["lat", "lon", f"CF_{group_name}"])
        
        # Save CSV
        df_final.to_csv(f"CF_files/CF_FWAC_{month_A}_{group_name}.csv", index=False)
        print(f"✅ File CF_FWAC_{month_A}_{group_name}.csv created :", df_final.shape)


✅ Fichier NE6D_CF_ACIDFW_01_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_01_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_01_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_02_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_02_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_02_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_03_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_03_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_03_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_04_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_04_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_04_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_05_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_05_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_05_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_06_nhx.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_06_nox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_06_sox.csv créé : (13104, 3)
✅ Fichier NE6D_CF_ACIDFW_07_